# Modelo de predicción de victoria antes de la partida
**Predicción de la probabilidad de victoria en League of Legends antes de iniciar la partida**
Inteligencia Artificial 1 · Universidad Industrial de Santander

Este notebook entrena y evalúa los modelos con el conjunto completo: 920 partidas válidas de la región LAN, cada una con
las diez partidas previas de sus diez jugadores. Las variables y el entrenamiento están implementados en el repositorio
`lol-win-predictor` (`src/features/build.py` y `src/models/train.py`); aquí se ejecutan y se presentan los resultados.

**Ejecución en Google Colab.** Suba los cinco archivos CSV del conjunto completo al panel de archivos de Colab y elija
*Entorno de ejecución → Ejecutar todas*. El notebook descarga el código del repositorio público. Si los CSV no están en el
panel, los busca en la carpeta `lol-win-predictor-data/completo` de Google Drive.

In [ ]:
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not Path('lol-win-predictor').exists():
        subprocess.run(['git', 'clone', '-q', 'https://github.com/JulianGomezPalencia/lol-win-predictor'], check=True)
    REPO = Path('lol-win-predictor')
else:
    REPO = Path('..')
sys.path.insert(0, str(REPO.resolve()))

from src.features import build
from src.models import train

# Los CSV se buscan junto al notebook, luego en Google Drive (en Colab) y por último en data/export del repositorio.
if Path('matches.csv').exists():
    DATA_DIR = Path('.')
elif IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = Path('/content/drive/MyDrive/lol-win-predictor-data/completo')
else:
    DATA_DIR = REPO / 'data' / 'export'
assert (DATA_DIR / 'matches.csv').exists(), f'No se encontró matches.csv en {DATA_DIR}'

plt.rcParams.update({'figure.figsize': (8, 4), 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.alpha': 0.3})
GRIS, ACENTO = '#9aa0a6', '#1a73e8'
print('Leyendo datos de', DATA_DIR)

## 1. Tabla de variables

Cada partida queda en una fila. Las variables son diferencias entre el equipo azul y el rojo, calculadas de dos formas:
como promedio de equipo (prefijo `dif_`) y jugador contra jugador en cada posición (por ejemplo `middle_dif_winrate_10`,
que resta el porcentaje de victorias del medio rojo al del medio azul). Las partidas previas reiniciadas no se cuentan y
ninguna partida previa es posterior a la partida que se predice.

In [ ]:
tabla = build.build(DATA_DIR)
variables = build.feature_columns(tabla)
print(f'{len(tabla)} partidas y {len(variables)} variables')
print(f'El equipo azul gana el {tabla["blue_win"].mean():.1%} de las partidas')
faltantes = tabla[variables].isna().sum()
print()
print('Variables con valores faltantes (jugadores sin historial; se imputan con la mediana de entrenamiento):')
print(faltantes[faltantes > 0].to_string())

## 2. Relación de cada variable con el resultado

Antes de entrenar se mide la correlación de cada variable con `blue_win`. Con 920 partidas, una correlación cuyo valor
absoluto supere 1,96/√920 ≈ 0,065 difícilmente se debe al azar; ese umbral aparece como línea punteada.

In [ ]:
corr = tabla[variables].corrwith(tabla['blue_win']).sort_values()
umbral = 1.96 / np.sqrt(len(tabla))
fig, ax = plt.subplots(figsize=(7, 8))
ax.barh(corr.index, corr.values, color=[ACENTO if abs(v) > umbral else GRIS for v in corr.values])
for x in (-umbral, umbral):
    ax.axvline(x, color='black', lw=1, ls='--')
ax.set_xlabel('Correlación con blue_win')
ax.set_title('Correlación de cada variable con la victoria del equipo azul (n = 920)')
ax.tick_params(axis='y', labelsize=8)
plt.tight_layout(); plt.show()
print('Por encima del umbral:')
print(corr[corr.abs() > umbral].round(3).to_string())

Con el conjunto completo aparece una señal que en la prueba piloto no se distinguía del ruido. La más clara es la
diferencia de KDA medio entre equipos: el equipo cuyos jugadores venían rindiendo mejor gana un poco más a menudo. Las
correlaciones son pequeñas, todas por debajo de 0,12, y con 29 variables es esperable que una o dos superen el umbral
por azar, así que conviene leerlas con cautela.

## 3. Partición y entrenamiento

Las partidas se ordenan por fecha: el 80 % más antiguo se usa para entrenar y el 20 % más reciente para evaluar. Los
hiperparámetros se eligen con validación cruzada temporal (`TimeSeriesSplit`, cinco cortes) dentro del entrenamiento,
y cada partida se pondera según su liga para que Maestro, Gran Maestro y Retador no dominen el ajuste.

In [ ]:
resultado = train.run(tabla)
entrenamiento, prueba = resultado['train'], resultado['test']
fecha = lambda ts: pd.to_datetime(ts, unit='ms', utc=True).strftime('%Y-%m-%d %H:%M')
print(f'Entrenamiento: {len(entrenamiento)} partidas, {fecha(entrenamiento.game_start_ts.min())} a {fecha(entrenamiento.game_start_ts.max())}')
print(f'Prueba: {len(prueba)} partidas, {fecha(prueba.game_start_ts.min())} a {fecha(prueba.game_start_ts.max())}')
assert entrenamiento.game_start_ts.max() <= prueba.game_start_ts.min()
print()
for nombre, busqueda in resultado['models'].items():
    print(f'{nombre}: {busqueda.best_params_}')

## 4. Resultados en el conjunto de prueba

Se comparan los dos modelos con dos líneas base: predecir siempre la clase más frecuente del entrenamiento y dar la
victoria al equipo con mejor porcentaje de victorias reciente. Además de la exactitud (*accuracy*) se reportan el log
loss y el puntaje de Brier, que evalúan la calidad de las probabilidades, y el área bajo la curva ROC. La segunda línea
base no produce probabilidades, solo 0 o 1, y por eso su log loss es muy alto.

In [ ]:
metricas = train.metrics_table(resultado['metrics'])
display(metricas)

fig, ax = plt.subplots(figsize=(7, 3.6))
nombres = metricas.index.tolist()
acc = metricas['accuracy'].values
err = [acc - metricas['ic95_bajo'].values, metricas['ic95_alto'].values - acc]
for k, (nombre, color) in enumerate(zip(nombres, [GRIS, GRIS, ACENTO, ACENTO])):
    ax.errorbar(acc[k], k, xerr=[[err[0][k]], [err[1][k]]], fmt='o', color=color, capsize=4, markersize=8)
ax.set_yticks(range(len(nombres)), nombres)
ax.axvline(0.5, color='black', lw=1, ls='--')
ax.set_xlim(0.3, 0.75); ax.set_xlabel('Accuracy en prueba (intervalo bootstrap del 95 %)')
ax.set_title(f'Exactitud de cada modelo ({len(prueba)} partidas de prueba)')
plt.tight_layout(); plt.show()

### Matriz de confusión y calibración

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
cm = resultado['metrics']['Regresión logística']['confusion']
axes[0].imshow(cm, cmap='Blues')
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, cm[i, j], ha='center', va='center', fontsize=12)
axes[0].set_xticks([0, 1], ['Predice rojo', 'Predice azul']); axes[0].set_yticks([0, 1], ['Gana rojo', 'Gana azul'])
axes[0].set_title('Matriz de confusión, regresión logística'); axes[0].grid(False)

from sklearn.calibration import calibration_curve
y_prueba = prueba['blue_win']
for nombre, color in [('Regresión logística', ACENTO), ('Bosque aleatorio', 'tab:orange')]:
    frac, media = calibration_curve(y_prueba, resultado['probabilities'][nombre], n_bins=5, strategy='quantile')
    axes[1].plot(media, frac, marker='o', color=color, label=nombre)
axes[1].plot([0, 1], [0, 1], color='black', lw=1, ls='--', label='Calibración perfecta')
axes[1].set_xlim(0.3, 0.7); axes[1].set_ylim(0.2, 0.8)
axes[1].set_xlabel('Probabilidad predicha de victoria azul'); axes[1].set_ylabel('Frecuencia observada')
axes[1].set_title('Curva de calibración (quintiles)'); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

for nombre in ('Regresión logística', 'Bosque aleatorio'):
    p = resultado['probabilities'][nombre]
    print(f'{nombre}: probabilidades entre {p.min():.2f} y {p.max():.2f}')

En la curva de calibración cada punto agrupa unas 37 partidas, por lo que es muy irregular. No se aprecia una
relación clara entre la probabilidad predicha y la frecuencia con que gana el azul, lo cual concuerda con un área bajo
la curva ROC apenas superior a 0,5 en este conjunto de prueba.

## 5. Variables más influyentes

Para la regresión logística se muestran los coeficientes, que son comparables entre sí porque las variables están
estandarizadas: un coeficiente positivo favorece al equipo azul. Para ambos modelos se calcula además la importancia
por permutación en el conjunto de prueba, es decir, cuánto empeora el log loss al desordenar cada variable.

In [ ]:
logit = resultado['models']['Regresión logística'].best_estimator_
coef = pd.Series(logit.named_steps['model'].coef_[0], index=resultado['features']).sort_values()
fig, ax = plt.subplots(figsize=(7, 8))
ax.barh(coef.index, coef.values, color=[ACENTO if v > 0 else GRIS for v in coef.values])
ax.axvline(0, color='black', lw=1)
ax.set_xlabel('Coeficiente (variables estandarizadas)')
ax.set_title('Coeficientes de la regresión logística'); ax.tick_params(axis='y', labelsize=8)
plt.tight_layout(); plt.show()

from sklearn.inspection import permutation_importance
X_prueba = prueba[resultado['features']]
for nombre, busqueda in resultado['models'].items():
    imp = permutation_importance(busqueda.best_estimator_, X_prueba, y_prueba, scoring='neg_log_loss',
                                 n_repeats=30, random_state=train.SEED)
    serie = pd.Series(imp.importances_mean, index=resultado['features']).sort_values(ascending=False)
    print(f'{nombre}, cinco variables más importantes (aumento del log loss al permutarlas):')
    print(serie.head(5).round(4).to_string()); print()

## 6. Validación temporal sobre todas las partidas

El conjunto de prueba tiene solo 184 partidas, jugadas en menos de un día, y su intervalo de confianza es amplio. Para
obtener una estimación más estable se recorre todo el periodo en cinco cortes: en cada uno, los modelos se entrenan con
las partidas anteriores al corte y predicen el bloque siguiente. Ninguna predicción usa partidas posteriores y en total
se evalúan unas 765 partidas.

In [ ]:
validacion = train.temporal_validation(tabla)
display(validacion)

## 7. Conclusiones

Con las 920 partidas el modelo encuentra una señal, aunque débil. En la validación temporal, con 765 predicciones hechas
siempre a partir de partidas anteriores, la regresión logística acierta el 54,1 % frente al 51,6 % de la línea base y
alcanza un área bajo la curva ROC de 0,56. En el conjunto de prueba final la diferencia es parecida (54,9 % contra
53,3 %), pero con 184 partidas queda dentro del intervalo de confianza y no basta para separar al modelo de la línea base.

Las variables que más aportan son el rendimiento reciente medido con el KDA, la comparación entre los dos soportes y
el tiempo desde la última partida. Según la importancia por permutación, la diferencia de KDA entre soportes es la que
más empeora el modelo cuando se desordena. Entre los coeficientes, el mayor corresponde al tiempo desde la última
partida: el equipo cuyos jugadores llevaban más tiempo sin jugar gana con algo menos de frecuencia. Otros coeficientes
tienen el signo contrario al esperado, como el porcentaje de victorias de los soportes o jugar en la posición habitual
en el medio, lo que indica que parte de estas relaciones son ruido.

El bosque aleatorio no supera a la regresión logística. Con tan pocas partidas y una señal tan tenue, un modelo más
flexible no encuentra interacciones útiles y tiende a ajustarse al ruido.

Para avanzar, la mejora con más probabilidad de éxito es ampliar el conjunto de datos, que hoy cubre solo dos semanas.
También convendría probar variables que la API no entrega de forma directa, como el rango de cada jugador en el momento
de la partida, que probablemente explica más que el historial reciente.